# SLM Backend — Layer 4 (Production) + Layer 5 (Personalization) + Layer 6 (Continuous Feedback)
Uses `slm_v1_adapter__2_.zip`. Run top to bottom in a T4 GPU runtime.

## Step 1 — Install dependencies

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio bitsandbytes peft transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 26.9 MB/s eta 0:00:00


## Step 2 — Set your ngrok auth token
Get one at https://dashboard.ngrok.com/tunnels/authtokens

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("PASTE_YOUR_OWN_NGROK_TOKEN_HERE")


## Step 3 — Upload and unzip the adapter
Upload `slm_v1_adapter__2_.zip`. Note: the zip's internal folder is named `slm_v1_adapter/` regardless of the zip's filename — confirm with `!ls -la` after unzipping.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload slm_v1_adapter__2_.zip

import zipfile, glob
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(".")

!ls -la


Saving slm_v1_adapter (2).zip to slm_v1_adapter (2).zip
total 44720
drwxr-xr-x 1 root root     4096 Aug 26 10:11  .
drwxr-xr-x 1 root root     4096 Aug 26 09:51  ..
drwxr-xr-x 4 root root     4096 Aug 24 13:27  .config
drwxr-xr-x 1 root root     4096 Aug 24 13:28  sample_data
drwxr-xr-x 2 root root     4096 Aug 26 10:11  slm_v1_adapter
-rw-r--r-- 1 root root 45769269 Aug 26 10:11 'slm_v1_adapter (2).zip'


## Optional: Set Hugging Face Token
If you encounter issues downloading models from Hugging Face Hub due to authentication or rate limits, you may need to set a Hugging Face token. You can generate one at [hf.co/settings/tokens](https://hf.co/settings/tokens).

In [ ]:
import os
from google.colab import userdata

# You can store your HF token as a Colab secret 'HF_TOKEN'
# Or uncomment the line below and paste it directly:
# os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("Colab Secret 'HF_TOKEN' not found. Please set it or uncomment the direct assignment.")
    HF_TOKEN = None

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("Hugging Face token loaded.")
else:
    print("Hugging Face token is not set. Model downloads may fail.")

Colab Secret 'HF_TOKEN' not found. Please set it or uncomment the direct assignment.
Hugging Face token is not set. Model downloads may fail.


In [ ]:
from huggingface_hub import login

# This will prompt you to enter your Hugging Face token
# You can generate one at https://huggingface.co/settings/tokens
login()

## Step 4 — Load base model (4-bit, fp16) + attach the adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

ADAPTER_PATH = "./slm_v1_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Base",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
model.config.use_cache = True
print("Model loaded.")


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded.


## Step 5 — Auth, rate limiting, EOS-mitigation truncation
(Layer 4 core utilities)

In [ ]:
import re
import time
from collections import defaultdict, deque

API_KEYS = {
    "dev-key-hr-001": {"employee_id": "E1001", "role": "HR"},
    "dev-key-dev-001": {"employee_id": "E1002", "role": "Developer"},
    "dev-key-pm-001": {"employee_id": "E1003", "role": "Product Manager"},
    "dev-key-devops-001": {"employee_id": "E1004", "role": "DevOps"},
    "dev-key-qa-001": {"employee_id": "E1005", "role": "QA & Testing"},
}

_request_log = defaultdict(deque)
MAX_REQUESTS = 10
WINDOW_SECONDS = 60

def check_rate_limit(key):
    now = time.time()
    log = _request_log[key]
    while log and now - log[0] > WINDOW_SECONDS:
        log.popleft()
    if len(log) >= MAX_REQUESTS:
        return False
    log.append(now)
    return True

def smart_truncate(text, expected_len_tokens=90, max_extra_ratio=1.5):
    """Mitigates the model's known EOS-learning gap: caps generation length
    relative to a target and trims to the last complete sentence."""
    max_tokens = int(expected_len_tokens * max_extra_ratio)
    tokens = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
        text = tokenizer.decode(tokens, skip_special_tokens=True)
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(sentences) > 1 and not sentences[-1].endswith((".", "!", "?")):
        text = " ".join(sentences[:-1])
    return text.strip()

print("Auth + rate limiter + truncation utilities ready.")


Auth + rate limiter + truncation utilities ready.


## Step 6 — Employee profile store (SQLite)
(Layer 5)

In [ ]:
import sqlite3
import json
from datetime import datetime

DB_PATH = "./employee_profiles.db"

def _get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    with _get_conn() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS profiles (
                employee_id     TEXT PRIMARY KEY,
                role            TEXT NOT NULL,
                response_style  TEXT DEFAULT 'balanced',
                verbosity       TEXT DEFAULT 'medium',
                language        TEXT DEFAULT 'english',
                focus_areas     TEXT DEFAULT '[]',
                updated_at      TEXT
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS memories (
                id              INTEGER PRIMARY KEY AUTOINCREMENT,
                employee_id     TEXT NOT NULL,
                summary         TEXT NOT NULL,
                approved        INTEGER DEFAULT 1,
                created_at      TEXT
            )
        """)
        conn.commit()
    print("Profile DB initialized.")

def upsert_profile(employee_id, role, preferences=None):
    prefs = preferences or {}
    with _get_conn() as conn:
        conn.execute("""
            INSERT INTO profiles (employee_id, role, response_style, verbosity, language, focus_areas, updated_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(employee_id) DO UPDATE SET
                role=excluded.role, response_style=excluded.response_style,
                verbosity=excluded.verbosity, language=excluded.language,
                focus_areas=excluded.focus_areas, updated_at=excluded.updated_at
        """, (employee_id, role,
              prefs.get("response_style", "balanced"),
              prefs.get("verbosity", "medium"),
              prefs.get("language", "english"),
              json.dumps(prefs.get("focus_areas", [])),
              datetime.utcnow().isoformat()))
        conn.commit()

def get_profile(employee_id):
    with _get_conn() as conn:
        row = conn.execute("SELECT * FROM profiles WHERE employee_id = ?", (employee_id,)).fetchone()
    if row is None:
        return None
    d = dict(row)
    d["focus_areas"] = json.loads(d["focus_areas"])
    return d

def add_memory(employee_id, summary, approved=True):
    with _get_conn() as conn:
        conn.execute("INSERT INTO memories (employee_id, summary, approved, created_at) VALUES (?, ?, ?, ?)",
                     (employee_id, summary, int(approved), datetime.utcnow().isoformat()))
        conn.commit()

def get_approved_memories(employee_id, limit=3):
    with _get_conn() as conn:
        rows = conn.execute("""SELECT summary FROM memories
            WHERE employee_id = ? AND approved = 1
            ORDER BY created_at DESC LIMIT ?""", (employee_id, limit)).fetchall()
    return [r["summary"] for r in rows]

def list_memories(employee_id):
    with _get_conn() as conn:
        rows = conn.execute("""SELECT id, summary, approved, created_at FROM memories
            WHERE employee_id = ? ORDER BY created_at DESC""", (employee_id,)).fetchall()
    return [dict(r) for r in rows]

def delete_memory(memory_id):
    with _get_conn() as conn:
        conn.execute("DELETE FROM memories WHERE id = ?", (memory_id,))
        conn.commit()

init_db()


Profile DB initialized.


## Step 7 — Context builder
(Layer 5 — prompt-side personalization, no model weights touched)

In [ ]:
VERBOSITY_MAP = {
    "brief":    "Keep your response concise — 2-3 sentences maximum.",
    "medium":   "Give a clear, complete response without unnecessary elaboration.",
    "detailed": "Provide a thorough, detailed response with full reasoning.",
}
STYLE_MAP = {
    "balanced":   "Balance directness with nuance.",
    "direct":     "Be direct and action-oriented. Lead with the recommendation.",
    "analytical": "Prioritize analysis and reasoning over direct recommendations.",
    "empathetic": "Prioritize understanding and tone before moving to action.",
}

def build_context(employee_id, role, instruction, context):
    profile = get_profile(employee_id)
    if profile is None:
        return {
            "system_prefix": f"You are a helpful {role} assistant at the company.",
            "instruction": instruction, "context": context, "profile_found": False,
        }
    verbosity_instr = VERBOSITY_MAP.get(profile["verbosity"], VERBOSITY_MAP["medium"])
    style_instr = STYLE_MAP.get(profile["response_style"], STYLE_MAP["balanced"])
    focus_areas = profile.get("focus_areas", [])
    focus_instr = f"\nThe employee's focus areas: {', '.join(focus_areas)}." if focus_areas else ""
    system_prefix = f"You are a helpful {role} assistant at the company.\n{style_instr} {verbosity_instr}{focus_instr}"
    memories = get_approved_memories(employee_id, limit=3)
    enriched_context = context
    if memories:
        memory_block = "\n".join(f"- {m}" for m in memories)
        enriched_context = f"[Past context:]\n{memory_block}\n\n{context}"
    return {
        "system_prefix": system_prefix, "instruction": instruction,
        "context": enriched_context, "profile_found": True,
    }

def build_prompt_from_context(ctx):
    return (
        f"<|system|>\n{ctx['system_prefix']}\n"
        f"<|user|>\n{ctx['instruction']}\n\nContext: {ctx['context']}\n"
        f"<|assistant|>\n"
    )

print("Context builder ready.")


Context builder ready.


HR policy rules

In [ ]:
HR_CASE_RULES = [
    (["harassment", "discrimination", "abuse"], "This requires formal HR or specialist review given the severity and compliance risk."),
    (["leave", "documentation", "medical"], "Verify documentation against policy; escalate to a specialist only if it deviates from standard requirements."),
    (["performance", "objectives", "underperforming"], "Use a structured, confidential discussion before considering formal action."),
    (["conflict", "disagreement", "manager"], "Address any immediate risk first, then facilitate a fair conversation between parties."),
    (["compensation", "pay", "salary"], "Escalate to a compensation specialist; avoid informal commitments."),
    (["grievance", "complaint"], "Preserve the case record and route to formal review if evidence or risk warrants it."),
    (["termination", "layoff", "dismissal"], "Requires formal HR and legal review before any action is communicated."),
    (["onboarding", "training gap"], "Lighter-touch coaching or clarification is usually appropriate here."),
]

def get_hr_guidance(instruction, context):
    text = (instruction + " " + context).lower()
    for keywords, guidance in HR_CASE_RULES:
        if any(kw in text for kw in keywords):
            return guidance
    return None

print("HR policy rules ready.")

HR policy rules ready.


## Step 8 — Interaction store + PII filtering
(Layer 6)

In [ ]:
import uuid
from datetime import timezone

INTERACTION_DB_PATH = "./interactions.db"

def _get_interaction_conn():
    conn = sqlite3.connect(INTERACTION_DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_interaction_db():
    with _get_interaction_conn() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS interactions (
                interaction_id   TEXT PRIMARY KEY,
                employee_id      TEXT NOT NULL,
                role             TEXT NOT NULL,
                prompt_raw       TEXT NOT NULL,
                prompt_filtered  TEXT NOT NULL,
                response_raw     TEXT NOT NULL,
                response_filtered TEXT NOT NULL,
                personalized     INTEGER,
                created_at       TEXT
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS feedback (
                id               INTEGER PRIMARY KEY AUTOINCREMENT,
                interaction_id   TEXT NOT NULL,
                employee_id      TEXT NOT NULL,
                thumbs           TEXT,
                stars            INTEGER,
                comment          TEXT,
                created_at       TEXT
            )
        """)
        conn.commit()
    print("Interaction DB initialized.")

# NOTE (v1 limitation): regex-based PII filtering catches structured patterns
# (emails, phones, SSN-like, employee-ID-like) but NOT unstructured PII such
# as names mentioned in free text - that needs an NER/ML-based approach.
PII_PATTERNS = [
    (re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"), "[EMAIL_REDACTED]"),
    (re.compile(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b"), "[PHONE_REDACTED]"),
    (re.compile(r"\b\d{3}-\d{2}-\d{4}\b"), "[SSN_REDACTED]"),
    (re.compile(r"\bE\d{4,}\b"), "[EMPLOYEE_ID_REDACTED]"),
]

def filter_pii(text):
    filtered = text
    for pattern, replacement in PII_PATTERNS:
        filtered = pattern.sub(replacement, filtered)
    return filtered

def log_interaction(employee_id, role, prompt, response, personalized):
    interaction_id = str(uuid.uuid4())
    with _get_interaction_conn() as conn:
        conn.execute("""
            INSERT INTO interactions
            (interaction_id, employee_id, role, prompt_raw, prompt_filtered,
             response_raw, response_filtered, personalized, created_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (interaction_id, employee_id, role, prompt, filter_pii(prompt),
              response, filter_pii(response), int(personalized),
              datetime.now(timezone.utc).isoformat()))
        conn.commit()
    return interaction_id

def save_feedback(interaction_id, employee_id, thumbs=None, stars=None, comment=None):
    with _get_interaction_conn() as conn:
        conn.execute("""
            INSERT INTO feedback (interaction_id, employee_id, thumbs, stars, comment, created_at)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (interaction_id, employee_id, thumbs, stars, comment,
              datetime.now(timezone.utc).isoformat()))
        conn.commit()

def get_interaction(interaction_id):
    with _get_interaction_conn() as conn:
        row = conn.execute("SELECT * FROM interactions WHERE interaction_id = ?", (interaction_id,)).fetchone()
    return dict(row) if row else None

def quality_check(interaction):
    flags = []
    resp = interaction["response_filtered"]
    if len(resp) < 20:
        flags.append("too_short")
    if len(resp) > 2000:
        flags.append("too_long")
    if "[EMAIL_REDACTED]" in resp or "[PHONE_REDACTED]" in resp or "[SSN_REDACTED]" in resp:
        flags.append("pii_was_present")
    words = resp.split()
    chunks = [" ".join(words[i:i+5]) for i in range(len(words) - 4)]
    if chunks and max((chunks.count(c) for c in set(chunks)), default=0) >= 3:
        flags.append("repetitive")
    return {"interaction_id": interaction["interaction_id"], "flags": flags, "needs_human_review": len(flags) > 0}

def run_batch_quality_filter(limit=100):
    with _get_interaction_conn() as conn:
        rows = conn.execute("SELECT * FROM interactions ORDER BY created_at DESC LIMIT ?", (limit,)).fetchall()
    results = [quality_check(dict(r)) for r in rows]
    flagged = [r for r in results if r["needs_human_review"]]
    print(f"Checked {len(results)} interactions, {len(flagged)} flagged for human review.")
    return results

init_interaction_db()


Interaction DB initialized.


## Step 9 — FastAPI app (Layer 4 + 5 + 6 combined)

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="SLM Backend - Layer 4+5+6")

class GenerateRequest(BaseModel):
    instruction: str
    context: str = ""

class GenerateResponse(BaseModel):
    role: str
    response: str
    personalized: bool
    interaction_id: str

class PreferencesModel(BaseModel):
    response_style: Optional[str] = "balanced"
    verbosity: Optional[str] = "medium"
    language: Optional[str] = "english"
    focus_areas: Optional[List[str]] = []

class ProfileRequest(BaseModel):
    preferences: PreferencesModel = PreferencesModel()

class MemoryRequest(BaseModel):
    summary: str
    approved: bool = True

class FeedbackRequest(BaseModel):
    interaction_id: str
    thumbs: Optional[str] = None
    stars: Optional[int] = None
    comment: Optional[str] = None

def _authenticate(x_api_key: str) -> dict:
    record = API_KEYS.get(x_api_key)
    if record is None:
        raise HTTPException(status_code=401, detail="Invalid API key.")
    return record

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest, x_api_key: str = Header(...)):
    record = _authenticate(x_api_key)
    if not check_rate_limit(record["employee_id"]):
        logger.warning(f"Rate limit exceeded for employee_id: {record['employee_id']}")
        raise HTTPException(status_code=429, detail="Rate limit exceeded.")
    if not req.instruction.strip():
        raise HTTPException(status_code=400, detail="instruction cannot be empty.")

    ctx = build_context(record["employee_id"], record["role"], req.instruction, req.context)
    prompt = build_prompt_from_context(ctx)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=600).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=200, do_sample=False,
            repetition_penalty=1.3, no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    response_text = smart_truncate(raw)

    interaction_id = log_interaction(
        employee_id=record["employee_id"], role=record["role"],
        prompt=req.instruction + " | context: " + req.context,
        response=response_text, personalized=ctx["profile_found"],
    )

    return GenerateResponse(role=record["role"], response=response_text,
        personalized=ctx["profile_found"], interaction_id=interaction_id)

@app.put("/profile")
def update_profile(req: ProfileRequest, x_api_key: str = Header(...)):
    record = _authenticate(x_api_key)
    upsert_profile(record["employee_id"], record["role"], req.preferences.model_dump())
    return {"message": "Profile updated.", "employee_id": record["employee_id"]}

@app.get("/profile")
def fetch_profile(x_api_key: str = Header(...)):
    record = _authenticate(x_api_key)
    profile = get_profile(record["employee_id"])
    return profile or {"message": "No profile yet.", "employee_id": record["employee_id"]}

@app.post("/memory")
def create_memory(req: MemoryRequest, x_api_key: str = Header(...)):
    record = _authenticate(x_api_key)
    if not req.summary.strip():
        raise HTTPException(status_code=400, detail="Memory summary cannot be empty.")
    add_memory(record["employee_id"], req.summary, req.approved)
    return {"message": "Memory saved."}

@app.get("/memories")
def fetch_memories(x_api_key: str = Header(...)):
    record = _authenticate(x_api_key)
    return {"memories": list_memories(record["employee_id"])}

@app.delete("/memory/{memory_id}")
def remove_memory(memory_id: int, x_api_key: str = Header(...)):
    _authenticate(x_api_key)
    delete_memory(memory_id)
    return {"message": f"Memory {memory_id} deleted."}

@app.post("/feedback")
def submit_feedback(req: FeedbackRequest, x_api_key: str = Header(...)):
    record = _authenticate(x_api_key)
    interaction = get_interaction(req.interaction_id)
    if interaction is None:
        raise HTTPException(status_code=404, detail="Interaction not found.")
    if interaction["employee_id"] != record["employee_id"]:
        raise HTTPException(status_code=403, detail="Cannot give feedback on another employee's interaction.")
    if req.thumbs is None and req.stars is None:
        raise HTTPException(status_code=400, detail="Provide at least thumbs or stars.")
    if req.thumbs is not None and req.thumbs not in ("up", "down"):
        raise HTTPException(status_code=400, detail="thumbs must be 'up' or 'down'.")
    if req.stars is not None and not (1 <= req.stars <= 5):
        raise HTTPException(status_code=400, detail="stars must be between 1 and 5.")
    save_feedback(req.interaction_id, record["employee_id"], req.thumbs, req.stars, req.comment)
    return {"message": "Feedback recorded.", "interaction_id": req.interaction_id}

print("App defined: /health /generate /profile /memory /memories /feedback")


App defined: /health /generate /profile /memory /memories /feedback


## Step 10 — Start the server + expose via ngrok

In [ ]:
import threading
import socket
import requests
import uvicorn
import nest_asyncio

nest_asyncio.apply()

def _find_free_port(start_port=8000, max_tries=10):
    for i in range(max_tries):
        port = start_port + i
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('localhost', port))
                return port
            except OSError:
                continue
    raise RuntimeError("Could not find a free port.")

def _run_uvicorn_server(port):
    config = uvicorn.Config(app, host="0.0.0.0", port=port, log_level="info")
    uvicorn.Server(config).run()

try:
    ngrok.kill()
except Exception:
    pass

_uvicorn_port = _find_free_port()
public_url = ngrok.connect(_uvicorn_port)
print("Public URL:", public_url)

threading.Thread(target=_run_uvicorn_server, args=(_uvicorn_port,), daemon=True).start()

time.sleep(3)
base_url = public_url.public_url
print("Health check:", requests.get(f"{base_url}/health").json())


INFO:     Started server process [1132]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://mongoose-glaring-bottom.ngrok-free.dev" -> "http://localhost:8000"
INFO:     34.50.177.246:0 - "GET /health HTTP/1.1" 200 OK
Health check: {'status': 'ok'}


## Step 11 — Test everything end-to-end (Layer 5 profile + Layer 6 logging/feedback)

In [ ]:
# 1. Set HR profile
r = requests.put(f"{base_url}/profile",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"preferences": {"response_style": "direct", "verbosity": "brief", "focus_areas": ["leave management", "grievances"]}})
print("Profile update:", r.status_code, r.json())

# 2. Add a memory
r = requests.post(f"{base_url}/memory",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"summary": "This employee prefers escalation to specialists for policy-grey-area cases."})
print("Memory save:", r.status_code, r.json())

# 3. Generate (personalized + auto-logged, includes a test email/phone to check PII filtering)
r = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Employee requests leave with non-standard documentation, contact them at jane.doe@company.com or 555-123-4567",
          "context": "Manager is unavailable for approval."})
gen_result = r.json()
print("Generate:", r.status_code, gen_result)

interaction_id = gen_result["interaction_id"]

# 4. Confirm PII was filtered in storage
stored = get_interaction(interaction_id)
print("\nStored prompt (raw):", stored["prompt_raw"])
print("Stored prompt (PII-filtered):", stored["prompt_filtered"])

# 5. Submit feedback
r = requests.post(f"{base_url}/feedback",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"interaction_id": interaction_id, "thumbs": "up", "comment": "Good, clear answer."})
print("\nFeedback:", r.status_code, r.json())

# 6. Fetch profile + memories to confirm persistence
print("\nProfile:", requests.get(f"{base_url}/profile", headers={"X-API-Key": "dev-key-hr-001"}).json())
print("Memories:", requests.get(f"{base_url}/memories", headers={"X-API-Key": "dev-key-hr-001"}).json())

# 7. Run the offline quality filter
run_batch_quality_filter(limit=20)


INFO:     34.50.177.246:0 - "PUT /profile HTTP/1.1" 200 OK


/tmp/ipykernel_1132/3369300797.py:52: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat()))


Profile update: 200 {'message': 'Profile updated.', 'employee_id': 'E1001'}
INFO:     34.50.177.246:0 - "POST /memory HTTP/1.1" 200 OK


/tmp/ipykernel_1132/3369300797.py:67: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (employee_id, summary, int(approved), datetime.utcnow().isoformat()))


Memory save: 200 {'message': 'Memory saved.'}
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 200 OK
Generate: 200 {'role': 'HR', 'response': 'Address the observable behavior first while preserving due-process safeguards against arbitrary changes. Reviewed the records, completed the behavioral step, documented the rationale, and communicated the defined next action to the appropriate stakeholder(s). A specific next action reduces uncertainty and shows that the unit was reviewed rather than ignored. Training should follow identified gaps instead of treatment alone. Limited the decision to its relevant facts and kept the proposed remedy explicit. Performance improvement requires measurable standards plus regular checks; random raises do not close causal loops. Must be explained separately from salary because it affects comparability across employees. Budget constraints limit discretionary spending on capital items this cycle but operational need remains valid.', 'personalized': Tru

[{'interaction_id': '9d856b89-7b0d-4206-a687-d022ad5b4462',
  'flags': [],
  'needs_human_review': False}]

## Step 12 — Test rate limiter (optional)

In [ ]:
for i in range(12):
    resp = requests.post(
        f"{base_url}/generate",
        headers={"X-API-Key": "dev-key-hr-001"},
        json={"instruction": "Quick test request", "context": "test"},
    )
    print(f"Request {i+1}: status={resp.status_code}")
    if resp.status_code == 429:
        print("  -> Rate limited:", resp.json())


INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 1: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 2: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 3: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 4: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 5: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 6: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 7: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 8: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 9: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 10: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 11: status=200
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 200 OK
Request 12: status=200


In [ ]:
print(_request_log)

defaultdict(<class 'collections.deque'>, {'E1001': deque([1787736461.3862646, 1787736482.1006637, 1787736501.7753844])})


In [ ]:
def check_rate_limit(key):
    now = time.time()
    log = _request_log[key]
    while log and now - log[0] > WINDOW_SECONDS:
        log.popleft()   # <- old timestamps get dropped here
    ...

In [ ]:
import concurrent.futures

def fire_request(i):
    resp = requests.post(f"{base_url}/generate",
        headers={"X-API-Key": "dev-key-hr-001"},
        json={"instruction": "Quick test request", "context": "test"})
    return i, resp.status_code

with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
    results = list(executor.map(fire_request, range(12)))

for i, status in sorted(results):
    print(f"Request {i+1}: status={status}")

INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 429 Too Many Requests


INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.50.177.246:0 - "POST /generate HTTP/1.1" 200 OK
Request 1: status=429
Request 2: status=200
Request 3: status=429
Request 4: status=200
Request 5: status=200
Request 6: status=200
Request 7: status=429
Request 8: status=200
Request 9: status=200
Request 10: status=200
Request 11: status=200
Request 12: status=200


In [ ]:
# Reset the rate limit log for a clean test
_request_log["E1001"].clear()

import concurrent.futures

def fire_request(i):
    resp = requests.post(f"{base_url}/generate",
        headers={"X-API-Key": "dev-key-hr-001"},
        json={"instruction": "Quick test request", "context": "test"})
    return i, resp.status_code

with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
    results = list(executor.map(fire_request, range(12)))

for i, status in sorted(results):
    print(f"Request {i+1}: status={status}")

INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests


INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests


INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests


INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests


INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests


INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests


INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests


INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
Request 1: status=429
Request 2: status=429
Request 3: status=429
Request 4: status=429
Request 5: status=429
Request 6: status=429
Request 7: status=429
Request 8: status=429
Request 9: status=429
Request 10: status=429
Request 11: status=429
Request 12: status=429


In [ ]:
# Reset rate limit for clean testing
_request_log["E1001"].clear()

# Test 1: should match the "harassment" rule -> guidance should push toward formal review
r1 = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Employee reports a harassment complaint against a colleague",
          "context": "Multiple incidents described, some witnesses available."})
print("=== Test 1: Harassment case (should match a rule) ===")
print(r1.json())
print()

# Test 2: should NOT match any rule -> falls back to model's own judgment (likely "coaching" default)
r2 = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Two employees disagree about project ownership credit",
          "context": "Both contributed significant work, no formal policy covers this scenario."})
print("=== Test 2: Unmatched case (no rule, model's own judgment) ===")
print(r2.json())
print()

# Test 3: should match "termination" rule -> should push toward formal HR+legal review
r3 = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Manager wants to terminate an employee for repeated policy violations",
          "context": "Employee has been on a performance improvement plan for two months."})
print("=== Test 3: Termination case (should match a rule) ===")
print(r3.json())

INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
=== Test 1: Harassment case (should match a rule) ===
{'detail': 'Rate limit exceeded.'}



INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
=== Test 2: Unmatched case (no rule, model's own judgment) ===
{'detail': 'Rate limit exceeded.'}



INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
=== Test 3: Termination case (should match a rule) ===
{'detail': 'Rate limit exceeded.'}


In [ ]:
print(dict(_request_log))

{'E1001': deque([])}


In [ ]:
_request_log.clear()  # clears ALL keys, safest full reset
print("Cleared. Current state:", dict(_request_log))

Cleared. Current state: {}


In [ ]:
# Reset rate limit for clean testing
_request_log["E1001"].clear()

# Test 1: should match the "harassment" rule -> guidance should push toward formal review
r1 = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Employee reports a harassment complaint against a colleague",
          "context": "Multiple incidents described, some witnesses available."})
print("=== Test 1: Harassment case (should match a rule) ===")
print(r1.json())
print()

# Test 2: should NOT match any rule -> falls back to model's own judgment (likely "coaching" default)
r2 = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Two employees disagree about project ownership credit",
          "context": "Both contributed significant work, no formal policy covers this scenario."})
print("=== Test 2: Unmatched case (no rule, model's own judgment) ===")
print(r2.json())
print()

# Test 3: should match "termination" rule -> should push toward formal HR+legal review
r3 = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Manager wants to terminate an employee for repeated policy violations",
          "context": "Employee has been on a performance improvement plan for two months."})
print("=== Test 3: Termination case (should match a rule) ===")
print(r3.json())

INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
=== Test 1: Harassment case (should match a rule) ===
{'detail': 'Rate limit exceeded.'}



INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
=== Test 2: Unmatched case (no rule, model's own judgment) ===
{'detail': 'Rate limit exceeded.'}



INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
=== Test 3: Termination case (should match a rule) ===
{'detail': 'Rate limit exceeded.'}


In [ ]:
r1 = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-dev-001"},  # switched key
    json={"instruction": "Employee reports a harassment complaint against a colleague",
          "context": "Multiple incidents described, some witnesses available."})
print(r1.json())

INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
{'detail': 'Rate limit exceeded.'}


In [ ]:
import time
print("Waiting 65 seconds for rate limit window to clear...")
time.sleep(65)
print("Done waiting.")

Waiting 65 seconds for rate limit window to clear...
Done waiting.


In [ ]:
r1 = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Employee reports a harassment complaint against a colleague",
          "context": "Multiple incidents described, some witnesses available."})
print("=== Test 1: Harassment case ===")
print(r1.json())

INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
=== Test 1: Harassment case ===
{'detail': 'Rate limit exceeded.'}


In [ ]:
print("Before:", dict(_request_log))

# Force-clear every key completely (not just clear the deque contents)
_request_log.clear()

print("After:", dict(_request_log))

Before: {'E1001': deque([]), 'E1002': deque([])}
After: {}


In [ ]:
r1 = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Employee reports a harassment complaint against a colleague",
          "context": "Multiple incidents described, some witnesses available."})
print(r1.json())

INFO:     34.21.222.47:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
{'detail': 'Rate limit exceeded.'}


Cell — Fresh dependency install, pinned versions (avoid the earlier binary conflict)

In [ ]:
!pip uninstall -y pyarrow datasets trl -q
!pip install -q "pyarrow==17.0.0" "datasets==3.0.1" "trl==0.11.4" "accelerate>=0.34.0"

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [ ]:
!pip install -q --upgrade pyarrow datasets
!pip install -q "trl==0.9.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 33.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 105.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 25.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.14.0.94 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but 

In [ ]:
# Uninstall to clear any conflicting versions
!pip uninstall -y trl datasets pyarrow accelerate -q

# Upgrade pip and setuptools for better dependency resolution
!pip install -q --upgrade pip setuptools

# Install the primary packages. Let pip manage the dependencies like datasets and pyarrow to ensure compatibility.
# This single command ensures pip tries its best to install trl 0.11.4 with compatible dependencies.
!pip install -q "trl==0.11.4" "accelerate>=0.34.0"

# Now import and check versions
import trl
print("trl:", trl.__version__)

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Try to import from trl again
# These were the lines that caused the error, so they are the final check.
from trl import PPOTrainer, PPOConfig
print("PPO imports OK")

from trl import RewardTrainer, RewardConfig
print("RewardTrainer imports OK")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-gbq 0.30.0 requires pyarrow>=4.0.0, which is not installed.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
trl: 0.9.6
torch: 2.11.0+cu128
CUDA available: True


RuntimeError: Failed to import trl.trainer.ppo_trainer because of the following error (look up to see its traceback):
pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility. Expected 112 from C header, got 104 from PyObject

Cell — Reload model + tokenizer + adapter (after restart)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

ADAPTER_PATH = "slm_v1_adapter" # Removed './' from the path

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Base", quantization_config=bnb_config,
    device_map="auto", torch_dtype=torch.float16, local_files_only=False
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, local_files_only=True)
print("Model loaded.")

OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

In [ ]:
!python -m venv /content/clean_env
!/content/clean_env/bin/pip install --upgrade pip


Error: Command '['/content/clean_env/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
/bin/bash: line 1: /content/clean_env/bin/pip: No such file or directory


In [ ]:
import sys
print("Python version:", sys.version)

import subprocess
print("\n--- apt / venv availability ---")
print(subprocess.run(["which", "python3"], capture_output=True, text=True).stdout)
print(subprocess.run(["python3", "-m", "venv", "--help"], capture_output=True, text=True).stdout[:200])

print("\n--- GPU ---")
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else "n/a")

print("\n--- Disk space ---")
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout)

print("\n--- Current package state ---")
try:
    import pyarrow
    print("pyarrow:", pyarrow.__version__)
except Exception as e:
    print("pyarrow import error:", e)
try:
    import trl
    print("trl:", trl.__version__)
except Exception as e:
    print("trl import error:", e)

Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]

--- apt / venv availability ---
/usr/bin/python3

usage: venv [-h] [--system-site-packages] [--symlinks | --copies] [--clear]
            [--upgrade] [--without-pip] [--prompt PROMPT] [--upgrade-deps]
            [--without-scm-ignore-files]
        

--- GPU ---
CUDA available: True
GPU: Tesla T4
VRAM (GB): 15.637086208

--- Disk space ---
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   56G   58G  50% /


--- Current package state ---
pyarrow: 18.1.0
trl: 0.9.6


In [ ]:
# Aggressively uninstall conflicting packages and setuptools
!pip uninstall -y trl datasets pyarrow accelerate setuptools -q

# Upgrade pip for better dependency resolution
!pip install -q --upgrade pip

# Force-downgrade setuptools to ensure compatibility with torch and other libraries
!pip install -q --force-reinstall "setuptools<82"

# Install the primary packages, letting pip handle dependencies like datasets and pyarrow
# This assumes a compatible set can be found for trl 0.11.4 with the current Python and setuptools.
!pip install -q "trl==0.11.4" "accelerate>=0.34.0"

# Now import and check versions
import trl
print("trl:", trl.__version__)

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Try to import from trl again as the final check
from trl import PPOTrainer, PPOConfig
print("PPO imports OK")

from trl import RewardTrainer, RewardConfig
print("RewardTrainer imports OK")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-gbq 0.30.0 requires pyarrow>=4.0.0, which is not installed.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
trl: 0.9.6
torch: 2.11.0+cu128
CUDA available: True


RuntimeError: Failed to import trl.trainer.ppo_trainer because of the following error (look up to see its traceback):
pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility. Expected 112 from C header, got 104 from PyObject

Layer 8 — Model Governance

In [ ]:
import json
import pandas as pd

def load_metrics(csv_path):
    return pd.read_csv(csv_path)

def compare_models(v1_metrics_path, v2_metrics_path, thresholds=None):
    """
    Compares two model versions on the metrics already computed in Layer 3.
    Returns a governance decision: IMPROVED / REGRESSION / MIXED.

    thresholds: dict of metric -> minimum acceptable delta (v2 - v1) to count
    as "improved" rather than "regression". Defaults are conservative.
    """
    if thresholds is None:
        thresholds = {
            "rouge1_f": 0.0,       # any non-negative change is fine
            "role_adherence": -0.02,  # allow small noise-level drops
            "length_ratio": None,   # closer to 1.0 is better, handled separately
        }

    v1 = load_metrics(v1_metrics_path)
    v2 = load_metrics(v2_metrics_path)

    v1_avg = v1.groupby("role")[["rouge1_f", "rougeL_f", "length_ratio", "role_adherence"]].mean()
    v2_avg = v2.groupby("role")[["rouge1_f", "rougeL_f", "length_ratio", "role_adherence"]].mean()

    report = {}
    overall_verdict = "IMPROVED"

    for role in v1_avg.index:
        if role not in v2_avg.index:
            continue
        rouge_delta = v2_avg.loc[role, "rouge1_f"] - v1_avg.loc[role, "rouge1_f"]
        adherence_delta = v2_avg.loc[role, "role_adherence"] - v1_avg.loc[role, "role_adherence"]
        length_v1 = abs(v1_avg.loc[role, "length_ratio"] - 1.0)
        length_v2 = abs(v2_avg.loc[role, "length_ratio"] - 1.0)
        length_improved = length_v2 < length_v1

        role_verdict = "IMPROVED"
        if adherence_delta < thresholds["role_adherence"]:
            role_verdict = "REGRESSION"
        elif rouge_delta < -0.02:
            role_verdict = "REGRESSION"

        report[role] = {
            "rouge1_delta": round(rouge_delta, 4),
            "role_adherence_delta": round(adherence_delta, 4),
            "length_ratio_v1": round(length_v1, 3),
            "length_ratio_v2": round(length_v2, 3),
            "length_improved": length_improved,
            "verdict": role_verdict,
        }

        if role_verdict == "REGRESSION":
            overall_verdict = "REGRESSION" if overall_verdict != "MIXED" else "MIXED"

    return {"overall_verdict": overall_verdict, "per_role": report}


def governance_decision(comparison_result):
    """Final staging decision based on the comparison."""
    verdict = comparison_result["overall_verdict"]
    if verdict == "IMPROVED":
        return "APPROVED -> Staging/Canary -> Production"
    elif verdict == "REGRESSION":
        return "REJECTED -> stays on current production version, flagged for review"
    else:
        return "MIXED -> manual review required before promotion"


# Example usage (once you have two metrics CSVs to compare):
# result = compare_models("layer3_metrics_v1.csv", "layer3_metrics_v2.csv")
# print(json.dumps(result, indent=2))
# print("\nDecision:", governance_decision(result))